# Warm up
Let's start with some warm up exercises to get you familiar with the database and do some SQL querying.
Make sure to have downloaded the NBA dataset form Kaggle as described in the README.

In [ ]:
import sqlite3 as sql
import pandas as pd
from pathlib import Path

## TODO: Use pathlib to get the path to the data directory. Path(__file__) won't work in a Jupyter notebook.
DATA_PATH = Path.cwd() / 'data'

## Create a SQLite connection and run some SQL queries

In [ ]:
con = sql.connect(DATA_PATH / "nba.sqlite")  # connect to the database
query = 'SELECT * FROM game LIMIT 10'  # write a query that selects the first 10 rows from the game table
top_10_game = pd.read_sql(query, con)  # read the query into a pandas dataframe
top_10_game.head()  # print the dataframe

In [ ]:
# let's look at the schema of the database
# TODO: sqlite_master is the name of the table you want to inspeact: Replace TABLE_NAME with the name of the table you want to inspect
query = """
    SELECT *
    FROM sqlite_master
"""

pd.read_sql(query, con).sort_values('name')

In [ ]:
# let's see all the tables we have in the database
# TODO: Filter for tables only
query = """
    SELECT name
    FROM sqlite_master
    WHERE type == 'table'
"""
pd.read_sql(query, con)

In [ ]:
# let's have a look at the columns in the game table
query = """
    PRAGMA table_info(draft_combine_stats)
"""
pd.read_sql(query, con)

In [ ]:
# let's find the earliest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date
    LIMIT 1
"""

print(pd.read_sql(query, con))

# let's find the latest recorded game
query = """
    SELECT game_date
    FROM game
    ORDER BY game_date DESC
    LIMIT 1
"""

print(pd.read_sql(query, con))


In [ ]:
# TODO: Find four interesting facts about the data and write a query to find the answer

In [ ]:
## Which team has scored the most points of all time?
query = """
SELECT team_name, SUM(pts) AS total_points, COUNT(*) AS games
FROM (
    SELECT team_name_home AS team_name, pts_home AS pts FROM game
    UNION ALL
    SELECT team_name_away, pts_away FROM game
)
WHERE team_name NOT LIKE '%All Star%'
GROUP BY team_name
ORDER BY total_points DESC
LIMIT 10
"""
print(pd.read_sql(query, con))

In [ ]:
## Is there a home-court advantage?
query = """
SELECT ROUND(100.0 * SUM(wl_home = 'W') / COUNT(*), 1) AS home_win_pct
FROM game
WHERE season_type = 'Regular Season'
"""
print(pd.read_sql(query, con))

In [ ]:
## How has the three-point shot grown over time?
query = """
SELECT strftime('%Y', game_date) AS year,
       ROUND(AVG(fg3a_home + fg3a_away), 1) AS avg_3pt_attempts_per_game
FROM game
WHERE season_type = 'Regular Season' AND fg3a_home IS NOT NULL
GROUP BY year
ORDER BY year
"""
print(pd.read_sql(query, con))

In [ ]:
## The biggest blowouts ever
query = """
SELECT game_date, team_name_home, pts_home, team_name_away, pts_away,
       ABS(pts_home - pts_away) AS margin
FROM game
WHERE season_type = 'Regular Season'
ORDER BY margin DESC
LIMIT 10
""" 
print(pd.read_sql(query, con))